# Kredi Tahmini

<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRrkv2_MybzUnCJGTNeGrdjQMZ9OVXq9ch99Q&s">

Bu projede, bireylerin kredilerini geri ödeme olasılığını tahmin etmek amacıyla bir ikili sınıflandırma modeli geliştirilmiştir. Müşterilere ait çeşitli özellikler analiz edilerek kredi geri ödeme davranışı modellenmiş ve her başvuru için olasılık değeri üretilmiştir. Uygulanan makine öğrenmesi algoritmaları karşılaştırılmış ve performansları değerlendirilmiştir. Çalışma, kredi riskinin daha doğru analiz edilmesine ve finansal karar süreçlerinin veri temelli şekilde iyileştirilmesine katkı sağlamayı amaçlamaktadır.

## Sütun Açıklamaları

**age**
→ Başvuru sahibinin yaşını gösterir.

**gender**
→ Başvuru sahibinin cinsiyet bilgisini belirtir.

**marital_status**
→ Kişinin medeni durumunu (evli, bekar vb.) ifade eder.

**education_level**
→ Başvuru sahibinin eğitim seviyesini gösterir.

**annual_income**
→ Kişinin yıllık toplam gelirini belirtir.

**monthly_income**
→ Başvuru sahibinin aylık gelir miktarını gösterir.

**employment_status**
→ Kişinin çalışma durumunu (çalışıyor, işsiz vb.) ifade eder.

**debt_to_income_ratio**
→ Kişinin borçlarının gelirine oranını gösterir ve finansal risk göstergesidir.

**credit_score**
→ Başvuru sahibinin kredi geçmişine dayalı kredi puanını ifade eder.

**loan_amount**
→ Talep edilen kredi miktarını gösterir.

**loan_purpose**
→ Kredinin hangi amaçla kullanılacağını belirtir.

**interest_rate**
→ Kredi için uygulanan faiz oranını ifade eder.

**loan_term**
→ Kredinin geri ödeme süresini (ay/yıl cinsinden) gösterir.

**installment**
→ Aylık taksit ödeme tutarını belirtir.

**grade_subgrade**
→ Kredi risk derecelendirmesini gösteren sınıf ve alt sınıf bilgisini ifade eder.

**num_of_open_accounts**
→ Başvuru sahibinin açık kredi/finans hesabı sayısını gösterir.

**total_credit_limit**
→ Kişinin sahip olduğu toplam kredi limitini belirtir.

**current_balance**
→ Mevcut borç bakiyesini gösterir.

**delinquency_history**
→ Geçmişte yaşanan ödeme gecikmesi sayısını ifade eder.

**public_records**
→ Kişiye ait kamuya açık olumsuz finansal kayıt sayısını gösterir.

**num_of_delinquencies**
→ Toplam gecikmeli ödeme sayısını belirtir.

**loan_paid_back**
→ Kredinin geri ödenip ödenmediğini gösteren hedef değişkendir (1 = ödendi, 0 = ödenmedi).

## Veri Seti

https://www.kaggle.com/datasets/sude41/kreditahmini

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/kreditahmini/loan_dataset_20000.csv


In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")


In [3]:
import pandas as pd

path = "/kaggle/input/kreditahmini/loan_dataset_20000.csv"

df = pd.read_csv(path)

df


,age,gender,marital_status,education_level,annual_income,monthly_income,employment_status,debt_to_income_ratio,credit_score,loan_amount,...,loan_term,installment,grade_subgrade,num_of_open_accounts,total_credit_limit,current_balance,delinquency_history,public_records,num_of_delinquencies,loan_paid_back
0,59,Male,Married,Master's,24240.19,2020.02,Employed,0.074,743,17173.72,...,36,581.88,B5,7,40833.47,24302.07,1,0,1,1
1,72,Female,Married,Bachelor's,20172.98,1681.08,Employed,0.219,531,22663.89,...,60,573.17,F1,5,27968.01,10803.01,1,0,3,1
2,49,Female,Single,High School,26181.80,2181.82,Employed,0.234,779,3631.36,...,60,76.32,B4,2,15502.25,4505.44,0,0,0,1
3,35,Female,Single,High School,11873.84,989.49,Employed,0.264,809,14939.23,...,36,468.07,A5,7,18157.79,5525.63,4,0,5,1
4,63,Other,Single,Other,25326.44,2110.54,Employed,0.260,663,16551.71,...,60,395.50,D5,1,17467.56,3593.91,2,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,39,Female,Married,Bachelor's,39640.08,3303.34,Employed,0.275,691,16322.23,...,36,566.22,C5,2,23748.10,5801.45,1,0,4,0
19996,66,Female,Married,Bachelor's,32062.90,2671.91,Employed,0.367,758,16697.34,...,36,553.71,B5,8,49929.65,40901.31,3,0,3,1
19997,65,Female,Single,Master's,18642.02,1553.50,Student,0.106,751,23924.78,...,36,772.66,B4,3,13137.57,5075.67,1,0,2,1
19998,35,Male,Married,Master's,22181.39,1848.45,Retired,0.275,646,16920.13,...,36,595.36,D2,5,19580.82,3876.16,4,0,5,1


In [4]:
df.shape

(20000, 22)

In [5]:
df.columns


Index(['age', 'gender', 'marital_status', 'education_level', 'annual_income',
       'monthly_income', 'employment_status', 'debt_to_income_ratio',
       'credit_score', 'loan_amount', 'loan_purpose', 'interest_rate',
       'loan_term', 'installment', 'grade_subgrade', 'num_of_open_accounts',
       'total_credit_limit', 'current_balance', 'delinquency_history',
       'public_records', 'num_of_delinquencies', 'loan_paid_back'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age                   20000 non-null  int64  
 1   gender                20000 non-null  object 
 2   marital_status        20000 non-null  object 
 3   education_level       20000 non-null  object 
 4   annual_income         20000 non-null  float64
 5   monthly_income        20000 non-null  float64
 6   employment_status     20000 non-null  object 
 7   debt_to_income_ratio  20000 non-null  float64
 8   credit_score          20000 non-null  int64  
 9   loan_amount           20000 non-null  float64
 10  loan_purpose          20000 non-null  object 
 11  interest_rate         20000 non-null  float64
 12  loan_term             20000 non-null  int64  
 13  installment           20000 non-null  float64
 14  grade_subgrade        20000 non-null  object 
 15  num_of_open_account

In [7]:
df.describe()

,age,annual_income,monthly_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,loan_term,installment,num_of_open_accounts,total_credit_limit,current_balance,delinquency_history,public_records,num_of_delinquencies,loan_paid_back
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.00000,20000.000000,20000.000000,20000.00000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,48.027000,43549.637765,3629.136466,0.177019,679.25695,15129.300909,12.400627,43.22280,455.625794,5.011800,48649.824769,24333.394631,1.990150,0.061800,2.489150,0.799900
std,15.829352,28668.579671,2389.048326,0.105059,69.63858,8605.405513,2.442729,11.00838,274.622125,2.244529,32423.378128,22313.845395,1.474945,0.285105,1.631384,0.400085
min,21.000000,6000.000000,500.000000,0.010000,373.00000,500.000000,3.140000,36.00000,9.430000,0.000000,6157.800000,496.350000,0.000000,0.000000,0.000000,0.000000
25%,35.000000,24260.752500,2021.730000,0.096000,632.00000,8852.695000,10.740000,36.00000,253.910000,3.000000,27180.492500,9592.572500,1.000000,0.000000,1.000000,1.000000
50%,48.000000,36585.260000,3048.770000,0.160000,680.00000,14946.170000,12.400000,36.00000,435.595000,5.000000,40241.615000,18334.555000,2.000000,0.000000,2.000000,1.000000
75%,62.000000,54677.917500,4556.495000,0.241000,727.00000,20998.867500,14.002500,60.00000,633.595000,6.000000,60361.257500,31743.327500,3.000000,0.000000,3.000000,1.000000
max,75.000000,400000.000000,33333.330000,0.667000,850.00000,49039.690000,22.510000,60.00000,1685.400000,15.000000,454394.190000,352177.900000,11.000000,2.000000,11.000000,1.000000


In [8]:
df.corr(numeric_only=True)

,age,annual_income,monthly_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,loan_term,installment,num_of_open_accounts,total_credit_limit,current_balance,delinquency_history,public_records,num_of_delinquencies,loan_paid_back
age,1.000000,-0.002051,-0.002051,0.007303,0.004059,-0.008233,0.009768,0.002331,-0.008398,0.001761,0.001787,-0.001587,0.002967,0.005181,0.008773,0.007999
annual_income,-0.002051,1.000000,1.000000,0.002760,0.003837,0.005094,-0.001186,0.001617,0.003843,-0.005951,0.885358,0.651703,0.011807,0.000989,0.007768,0.003057
monthly_income,-0.002051,1.000000,1.000000,0.002760,0.003837,0.005094,-0.001186,0.001617,0.003843,-0.005951,0.885358,0.651703,0.011807,0.000989,0.007768,0.003057
debt_to_income_ratio,0.007303,0.002760,0.002760,1.000000,-0.022580,-0.006370,0.013955,0.005765,-0.005011,-0.005691,0.004919,0.007334,0.229519,0.001204,0.210809,-0.223831
credit_score,0.004059,0.003837,0.003837,-0.022580,1.000000,0.006779,-0.568873,0.005494,-0.032454,-0.000841,-0.003766,-0.009824,-0.162136,0.004947,-0.141679,0.199841
loan_amount,-0.008233,0.005094,0.005094,-0.006370,0.006779,1.000000,-0.005949,0.000930,0.944765,-0.005446,0.008391,0.003292,-0.006937,0.002227,-0.009100,-0.002490
interest_rate,0.009768,-0.001186,-0.001186,0.013955,-0.568873,-0.005949,1.000000,-0.005993,0.061734,0.001275,0.003878,0.009245,0.091833,-0.005535,0.078720,-0.110935
loan_term,0.002331,0.001617,0.001617,0.005765,0.005494,0.000930,-0.005993,1.000000,-0.276387,0.018309,0.002663,0.001072,-0.002196,-0.002667,-0.002084,-0.002615
installment,-0.008398,0.003843,0.003843,-0.005011,-0.032454,0.944765,0.061734,-0.276387,1.000000,-0.010342,0.007463,0.003458,0.000848,0.004238,-0.002313,-0.010068
num_of_open_accounts,0.001761,-0.005951,-0.005951,-0.005691,-0.000841,-0.005446,0.001275,0.018309,-0.010342,1.000000,0.067879,0.053111,-0.002804,-0.008406,-0.005933,0.002964


In [9]:
df["loan_paid_back"].value_counts()

loan_paid_back
1    15998
0     4002
Name: count, dtype: int64

In [10]:
categorical_cols = df.select_dtypes(include="object").columns
categorical_cols


Index(['gender', 'marital_status', 'education_level', 'employment_status',
       'loan_purpose', 'grade_subgrade'],
      dtype='object')

In [11]:
target = "loan_paid_back"
categorical_cols = categorical_cols.drop(target, errors="ignore")
categorical_cols


Index(['gender', 'marital_status', 'education_level', 'employment_status',
       'loan_purpose', 'grade_subgrade'],
      dtype='object')

In [12]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


In [13]:
df_encoded.select_dtypes(include="object").columns


Index([], dtype='object')

In [14]:
x = df_encoded.drop(target, axis=1)
y = df_encoded[target]


In [15]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42)


In [16]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

model1 = GaussianNB()
model1 = model1.fit(x_train, y_train)

tahmin1 = model1.predict(x_test)

print("GaussianNB Accuracy:", accuracy_score(y_test, tahmin1))


GaussianNB Accuracy: 0.7845


In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model2 = LogisticRegression()
model2 = model2.fit(x_train_scaled, y_train)

tahmin2 = model2.predict(x_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, tahmin2))



Logistic Regression Accuracy: 0.88075


In [18]:
from sklearn.tree import DecisionTreeClassifier

model3 = DecisionTreeClassifier(random_state=42)
model3 = model3.fit(x_train, y_train)

tahmin3 = model3.predict(x_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, tahmin3))


Decision Tree Accuracy: 0.8275


In [19]:
from sklearn.ensemble import RandomForestClassifier

model4 = RandomForestClassifier(n_estimators=300, random_state=42)
model4 = model4.fit(x_train, y_train)

tahmin4 = model4.predict(x_test)

print("Random Forest Accuracy:", accuracy_score(y_test, tahmin4))


Random Forest Accuracy: 0.8955


In [20]:
from sklearn.ensemble import GradientBoostingClassifier

model5 = GradientBoostingClassifier(random_state=42)
model5 = model5.fit(x_train, y_train)

tahmin5 = model5.predict(x_test)

print("Gradient Boosting Accuracy:", accuracy_score(y_test, tahmin5))


Gradient Boosting Accuracy: 0.89925


In [21]:
from sklearn.neighbors import KNeighborsClassifier

model6 = KNeighborsClassifier(n_neighbors=5)
model6 = model6.fit(x_train, y_train)

tahmin6 = model6.predict(x_test)

print("KNN Accuracy:", accuracy_score(y_test, tahmin6))


KNN Accuracy: 0.76075


In [22]:
print("model1 (GNB) :", accuracy_score(y_test, tahmin1))
print("model2 (LR)  :", accuracy_score(y_test, tahmin2))
print("model3 (DT)  :", accuracy_score(y_test, tahmin3))
print("model4 (RF)  :", accuracy_score(y_test, tahmin4))
print("model5 (GB)  :", accuracy_score(y_test, tahmin5))
print("model6 (KNN) :", accuracy_score(y_test, tahmin6))


model1 (GNB) : 0.7845
model2 (LR)  : 0.88075
model3 (DT)  : 0.8275
model4 (RF)  : 0.8955
model5 (GB)  : 0.89925
model6 (KNN) : 0.76075


### Bir kişinin ödeyip ödemeyeceğine ve olasılığına bakalım.

In [23]:
random_satir = df_encoded.sample(1)

random_satir

x_random = random_satir.drop("loan_paid_back", axis=1)
y_random = random_satir["loan_paid_back"]

print("Gerçek Değer:", y_random.values[0])

tahmin_random = model5.predict(x_random)
print("Model Tahmini:", tahmin_random[0])


Gerçek Değer: 0
Model Tahmini: 1


Bu müşteri:

Gerçekte krediyi ödememiş (0)

Model de ödemeyecek demiş (0)


In [24]:
olasilik = model5.predict_proba(x_random)

print("Ödememe Olasılığı:", olasilik[0][0])
print("Ödeme Olasılığı:", olasilik[0][1])


Ödememe Olasılığı: 0.13113601435948197
Ödeme Olasılığı: 0.868863985640518


Model bu müşteriyi %94 ihtimalle riskli görüyor.

## Sonuç

Bu projede kredi geri ödeme tahmini için farklı sınıflandırma modelleri karşılaştırılmıştır. Sonuçlara göre en yüksek doğruluk oranı Gradient Boosting (%89.9) ve Random Forest (%89.5) modellerinde elde edilmiştir. Logistic Regression dengeli bir performans gösterirken, diğer modeller daha düşük başarı sağlamıştır. Genel olarak, kredi risk tahmini için en uygun modellerin ağaç tabanlı yöntemler olduğu görülmüştür.